# Multiple Linear Regression - Toyota Corolla
This notebook performs EDA, preprocessing, builds several MLR models (Linear, Reduced OLS, Lasso, Ridge), evaluates performance, and interprets coefficients.

In [ ]:
# Cell 1: Import libraries
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# Cell 2: Load dataset
csv_path = 'ToyotaCorolla - MLR.csv'  # ensure this file is in the same folder
df = pd.read_csv(csv_path)
print(df.shape)
df.head()

In [ ]:
# Cell 3: Basic info
print(df.dtypes)
print(df.isnull().sum())
df.describe().T

In [ ]:
# Cell 4: Preprocessing
data = df.copy()
data.columns = [c.strip() for c in data.columns]

# Convert Doors if not numeric
if 'Doors' in data.columns:
    data['Doors_original'] = data['Doors']
    data['Doors'] = pd.to_numeric(data['Doors'], errors='coerce').fillna(data['Doors'].median())

# Ensure Automatic is numeric
if 'Automatic' in data.columns and data['Automatic'].dtype == object:
    data['Automatic'] = data['Automatic'].str.lower().map({'yes':1,'no':0,'true':1,'false':0})
data['Automatic'] = pd.to_numeric(data['Automatic'], errors='coerce').fillna(0).astype(int)

# Encode Fuel_Type if present
if 'Fuel_Type' in data.columns:
    data = pd.concat([data, pd.get_dummies(data['Fuel_Type'], prefix='Fuel', drop_first=True)], axis=1)

# Target variable
data['Price'] = pd.to_numeric(data['Price'], errors='coerce')

# Fill numeric NaNs with median
for c in data.select_dtypes(include=[np.number]).columns:
    data[c] = data[c].fillna(data[c].median())

print(data.shape)
data.head()

In [ ]:
# Cell 5: Correlation matrix
corr = data.corr()
plt.figure(figsize=(8,6))
plt.imshow(corr, aspect='auto')
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Correlation matrix')
plt.show()
corr['Price'].abs().sort_values(ascending=False).head(10)

In [ ]:
# Cell 6: Train/test split
drop_cols = [c for c in ['Fuel_Type','Doors_original'] if c in data.columns]
X = data.drop(columns=drop_cols+['Price']).select_dtypes(include=[np.number])
y = data['Price']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print(X_train.shape, X_test.shape)

In [ ]:
# Cell 7: Helper metrics function
def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE(%)': mape}

In [ ]:
# Cell 8: Model 1 - LinearRegression (all features)
lr = LinearRegression().fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
metrics_lr = regression_metrics(y_test, y_pred_lr)
print(metrics_lr)
pd.DataFrame({'feature':X.columns,'coef':lr.coef_}).sort_values(by='coef', key=abs)

In [ ]:
# Cell 9: Model 2 - OLS with backward elimination
selected = X_train.columns.tolist()
while True:
    model_sm = sm.OLS(y_train, sm.add_constant(X_train[selected], has_constant='add')).fit()
    pvals = model_sm.pvalues.drop('const', errors='ignore')
    if pvals.empty: break
    max_p = pvals.max()
    if max_p > 0.05:
        feature_remove = pvals.idxmax()
        print("Removing", feature_remove)
        selected.remove(feature_remove)
        if not selected: break
    else: break

print("Final selected:", selected)
print(model_sm.summary())

lr_red = LinearRegression().fit(X_train[selected], y_train)
y_pred_red = lr_red.predict(X_test[selected])
metrics_red = regression_metrics(y_test, y_pred_red)
print(metrics_red)

In [ ]:
# Cell 10: Model 3 - LassoCV
lasso = Pipeline([('scaler', StandardScaler()), ('lasso', LassoCV(cv=5, random_state=42, max_iter=10000))])
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
print(regression_metrics(y_test, y_pred_lasso))
pd.Series(lasso.named_steps['lasso'].coef_, index=X.columns)[lambda s:s!=0]

In [ ]:
# Cell 11: Model 4 - RidgeCV
alphas = np.logspace(-3,3,50)
ridge = Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=alphas, cv=5))])
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
print(regression_metrics(y_test, y_pred_ridge))

In [ ]:
# Cell 12: Compare models
results = pd.DataFrame([
    {'model':'Linear (all)', **metrics_lr},
    {'model':'Linear (reduced)', **metrics_red},
    {'model':'Lasso', **regression_metrics(y_test,y_pred_lasso)},
    {'model':'Ridge', **regression_metrics(y_test,y_pred_ridge)}
])
results

In [ ]:
# Cell 13: Residuals plot for best model
best = results.loc[results['RMSE'].idxmin(),'model']
print("Best:", best)
if best=='Linear (all)': y_pred_best=y_pred_lr
elif best=='Linear (reduced)': y_pred_best=y_pred_red
elif best=='Lasso': y_pred_best=y_pred_lasso
else: y_pred_best=y_pred_ridge

resid = y_test - y_pred_best
plt.scatter(y_pred_best, resid)
plt.axhline(0)
plt.xlabel('Predicted'); plt.ylabel('Residuals')
plt.title(best)
plt.show()

In [ ]:
# Cell 14: VIF for reduced model
feat = selected if selected else X.columns.tolist()
X_vif = X[feat].assign(const=1)
vif_data = [{'feature':f, 'VIF':variance_inflation_factor(X_vif.values,i)}
            for i,f in enumerate(X_vif.columns[:-1])]
pd.DataFrame(vif_data).sort_values('VIF',ascending=False)